In [1]:
%pip install -q timm scikit-learn

In [2]:
from pathlib import Path

DATA_DIR = Path("/content/itmo-cv/lab-4")
DVM_ZIP = DATA_DIR / "resized_DVM_v2.zip"
DVM_ROOT = DATA_DIR / "resized_DVM_v2"
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
import subprocess
import zipfile

FIGSHARE_FILE_ID = "34792453"
FIGSHARE_URL = f"https://ndownloader.figshare.com/files/{FIGSHARE_FILE_ID}"

if DVM_ZIP.is_file() and not zipfile.is_zipfile(DVM_ZIP):
    DVM_ZIP.unlink()
if not DVM_ZIP.is_file():
    print(f"Downloading to {DVM_ZIP} ...")
    subprocess.run(
        [
            "wget",
            "--continue",
            "--tries=20",
            "--read-timeout=300",
            "--user-agent=Mozilla/5.0",
            "-O",
            str(DVM_ZIP),
            FIGSHARE_URL,
        ],
        check=True,
    )
if not zipfile.is_zipfile(DVM_ZIP):
    raise RuntimeError(
        f"{DVM_ZIP} is not a valid zip; delete it and rerun (often HTML or truncated download)."
    )

In [18]:
import io
import random
import zipfile
from collections import Counter

import numpy as np
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TOP_K = 8
MAX_PER_CLASS = 500
VAL_FRAC = 0.2
IMG_SIZE = 224
BATCH = 32
EPOCHS_FT = 6
EPOCHS_VIT = 12
EPOCHS_SCRATCH = 14
LR_FT = 3e-4
LR_HEAD = 1e-3
LR_VIT_HEAD = 5e-4
LR_VIT_BACK = 2e-5
LR_SCRATCH = 1e-3

In [13]:
def collect_colour_paths_dir(root: Path):
    exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
    out = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in exts:
            out.append((str(p), p.parent.name))
    return out


def collect_members_zip(zip_path: Path):
    exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = [n for n in zf.namelist() if not n.endswith("/")]
    return [(n, Path(n).parent.name) for n in names if Path(n).suffix in exts]


has_dir = DVM_ROOT.is_dir() and next(DVM_ROOT.rglob("*.jpg"), None) is not None
if has_dir:
    raw = collect_colour_paths_dir(DVM_ROOT)
    dvm_zip_for_ds = None
else:
    raw = collect_members_zip(DVM_ZIP)
    dvm_zip_for_ds = DVM_ZIP
counts = Counter(l for _, l in raw)
keep_labels = {c for c, _ in counts.most_common(TOP_K)}
filtered = [(p, l) for p, l in raw if l in keep_labels]
by_c = {}
for p, l in filtered:
    by_c.setdefault(l, []).append(p)
balanced = []
for l, paths in by_c.items():
    rng = random.Random(seed)
    paths = paths[:]
    rng.shuffle(paths)
    take = paths[: min(MAX_PER_CLASS, len(paths))]
    balanced.extend([(p, l) for p in take])
label_names = sorted(keep_labels)
label_to_idx = {n: i for i, n in enumerate(label_names)}
paths = [p for p, _ in balanced]
y = [label_to_idx[l] for _, l in balanced]
train_paths, val_paths, y_train, y_val = train_test_split(
    paths, y, test_size=VAL_FRAC, random_state=seed, stratify=y
)
len(train_paths), len(val_paths), len(label_names)

(2560, 640, 8)

In [14]:
class ColourDS(Dataset):
    def __init__(self, items, labels, tfm, zip_path=None):
        self.items = items
        self.labels = labels
        self.tfm = tfm
        self.zip_path = zip_path
        self._zf = zipfile.ZipFile(zip_path, "r") if zip_path is not None else None

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        if self._zf is not None:
            with self._zf.open(self.items[i]) as f:
                img = Image.open(io.BytesIO(f.read())).convert("RGB")
        else:
            img = Image.open(self.items[i]).convert("RGB")
        x = transforms.functional.pil_to_tensor(img).float() / 255.0
        x = self.tfm(x)
        return x, self.labels[i]

    def __del__(self):
        if self._zf is not None:
            self._zf.close()


tfm_train = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.05, 0.05, 0.03, 0.02),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
    ]
)
tfm_val = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
    ]
)

train_ds = ColourDS(train_paths, y_train, tfm_train, dvm_zip_for_ds)
val_ds = ColourDS(val_paths, y_val, tfm_val, dvm_zip_for_ds)
_pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=_pin)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=_pin)

In [15]:
@torch.no_grad()
def eval_f1(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()
        ys.append(y.numpy())
        ps.append(pred)
    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    return float(f1_score(ys, ps, average="macro"))


def train_epochs(model, loader, opt, crit, epochs, scheduler=None):
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = crit(logits, y)
            loss.backward()
            opt.step()
        if scheduler is not None:
            scheduler.step()


def vit_adamw(model, lr_head, lr_back):
    head_p, back_p = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_p if n.startswith("head") else back_p).append(p)
    return torch.optim.AdamW(
        [{"params": back_p, "lr": lr_back}, {"params": head_p, "lr": lr_head}],
        weight_decay=0.05,
    )

In [16]:
n_cls = len(label_names)
crit = nn.CrossEntropyLoss()
results = {}

m1 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
m1.fc = nn.Linear(m1.fc.in_features, n_cls)
m1 = m1.to(device)
back_p = [p for n, p in m1.named_parameters() if not n.startswith("fc")]
head_p = [p for n, p in m1.named_parameters() if n.startswith("fc")]
opt1 = torch.optim.AdamW(
    [{"params": back_p, "lr": LR_FT * 0.2}, {"params": head_p, "lr": LR_HEAD}],
    weight_decay=1e-4,
)
sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=EPOCHS_FT)
train_epochs(m1, train_loader, opt1, crit, EPOCHS_FT, sch1)
results["resnet18_imagenet1k"] = eval_f1(m1, val_loader)
del m1, opt1
torch.cuda.empty_cache()

In [19]:
m2 = timm.create_model(
    "vit_small_patch14_dinov2.lvd142m",
    pretrained=True,
    num_classes=n_cls,
    img_size=224,
)
m2 = m2.to(device)
opt2 = vit_adamw(m2, LR_VIT_HEAD, LR_VIT_BACK)
sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS_VIT)
train_epochs(m2, train_loader, opt2, crit, EPOCHS_VIT, sch2)
results["vit_small_dinov2"] = eval_f1(m2, val_loader)
del m2, opt2
torch.cuda.empty_cache()

In [20]:
class SmallCNN(nn.Module):
    def __init__(self, n_cls):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(128, n_cls)

    def forward(self, x):
        x = self.f(x).flatten(1)
        return self.fc(x)


m3 = SmallCNN(n_cls).to(device)
opt3 = torch.optim.AdamW(m3.parameters(), lr=LR_SCRATCH)
train_epochs(m3, train_loader, opt3, crit, EPOCHS_SCRATCH)
results["small_cnn_scratch"] = eval_f1(m3, val_loader)

In [21]:
for k, v in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"{k}: F1_macro={v:.4f}")

vit_small_dinov2: F1_macro=0.6740
resnet18_imagenet1k: F1_macro=0.6511
small_cnn_scratch: F1_macro=0.4033
